
# Orbital Insight — Cloudflare + Collision Timer Notebook

Run all cells top to bottom.

After the Cloudflare Tunnel cell prints a public URL, open:

`<public_url>/ui`


In [1]:
!pip install -q fastapi uvicorn nest_asyncio requests


In [2]:
app_code = r'''

from fastapi import FastAPI
from fastapi.responses import HTMLResponse, JSONResponse, RedirectResponse
import uvicorn, time, requests, math

app = FastAPI(title="Orbital Insight Final Corrected Demo")

CONFIG = {"sat_count": 5, "debris_count": 4, "orbit_radius_km": 7700, "alert_topic": ""}
MISSION = {"started": False, "start_time": None, "alert_sent": False}
EVENTS = []

def log_event(kind, msg):
    EVENTS.append({"kind": kind, "message": msg, "ts": time.time()})
    if len(EVENTS) > 120:
        del EVENTS[:-120]

def send_alert(title, body):
    topic = CONFIG.get("alert_topic", "").strip()
    if not topic:
        return {"sent": False, "reason": "No topic configured"}
    try:
        r = requests.post(
            f"https://ntfy.sh/{topic}",
            data=body.encode("utf-8"),
            headers={"Title": title, "Priority": "urgent", "Tags": "warning,rocket,satellite"},
            timeout=6,
        )
        return {"sent": bool(r.ok), "status_code": r.status_code, "reason": r.text[:200], "topic": topic}
    except Exception as e:
        return {"sent": False, "reason": str(e), "topic": topic}



def mission_objects_for_phase_py(phase, elapsed, config):
    out = []
    sat_count = min(5, max(2, int(config.get("sat_count", 5))))
    connected = max(1, sat_count - 2)
    disconnected = sat_count - connected

    for i in range(connected):
        a = (i / connected) * math.pi * 2 + elapsed * 0.23
        r = 9 + (i % 2) * 0.5
        out.append({"id": f"SAT-C{i+1}", "type": "sat", "connected": True, "x": math.cos(a) * r, "y": 0.0, "z": math.sin(a) * r})

    for i in range(disconnected):
        a = 1.2 + i * 1.4 + elapsed * 0.19
        r = 10.2
        if phase in ("threat", "avoidance", "recovery", "stable"):
            r -= min(2.5, max(0.0, elapsed - 24.0) * 0.11)
        out.append({"id": f"SAT-D{i+1}", "type": "sat", "connected": False, "x": math.cos(a) * r, "y": 0.0, "z": math.sin(a) * r})

    for i in range(max(1, int(config.get("debris_count", 4)))):
        a = 0.8 + i * 0.9 + elapsed * (0.16 + i * 0.01)
        r = 11.2 + i * 0.35
        if phase in ("threat", "avoidance", "recovery", "stable"):
            r -= min(3.0, max(0.0, elapsed - 24.0) * 0.08)
        out.append({"id": f"DEB-{i+1}", "type": "debris", "connected": False, "x": math.cos(a) * r, "y": 0.0, "z": math.sin(a) * r})

    protected_sat = next((o for o in out if o["id"] == "SAT-C1"), None)
    if protected_sat:
        if phase == "avoidance":
            protected_sat["y"] = min(2.2, max(0.0, elapsed - 34.0) * 0.42)
        if phase == "recovery":
            protected_sat["y"] = max(0.0, 2.2 - max(0.0, elapsed - 44.0) * 0.24)
    return out


def collision_metrics(phase, elapsed, config):
    km_per_unit = float(config.get("orbit_radius_km", 7700.0)) / 9.0

    def nearest_threat_at(sample_phase, sample_elapsed):
        objs = mission_objects_for_phase_py(sample_phase, sample_elapsed, config)
        protected = next((o for o in objs if o["id"] == "SAT-C1"), None)
        threats = [o for o in objs if o["id"].startswith("SAT-D") or o["id"].startswith("DEB-")]
        if not protected or not threats:
            return None
        best = None
        for t in threats:
            d = math.sqrt((protected["x"] - t["x"])**2 + (protected["y"] - t["y"])**2 + (protected["z"] - t["z"])**2)
            if best is None or d < best["distance_units"]:
                best = {"threat_id": t["id"], "distance_units": d}
        return best

    if phase in ("overview", "launch", "deploy", "orbit_nominal"):
        current = nearest_threat_at("orbit_nominal", max(elapsed, 14.0))
        return {
            "status": "Monitoring",
            "time_to_collision_s": None,
            "distance_left_km": None,
            "closest_approach_km": round(current["distance_units"] * km_per_unit, 1) if current else None,
            "tracked_object": current["threat_id"] if current else None,
            "note": "No imminent collision window yet."
        }

    sample_end = 34.0 if phase == "threat" else min(max(elapsed + 8.0, elapsed), 44.0)
    best_future = None
    t = elapsed
    while t <= sample_end + 1e-9:
        sample_phase = phase
        if t < 34.0:
            sample_phase = "threat"
        elif t < 44.0:
            sample_phase = "avoidance"
        else:
            sample_phase = "recovery"
        current = nearest_threat_at(sample_phase, t)
        if current and (best_future is None or current["distance_units"] < best_future["distance_units"]):
            best_future = {**current, "sample_elapsed": t}
        t += 0.25

    current = nearest_threat_at(phase, elapsed)
    if not current:
        return {
            "status": "Unavailable",
            "time_to_collision_s": None,
            "distance_left_km": None,
            "closest_approach_km": None,
            "tracked_object": None,
            "note": "Collision telemetry unavailable."
        }

    if phase == "threat":
        status = "Threat detected"
        note = "Countdown is based on predicted closest approach before avoidance burn."
        time_to_collision = max(0.0, best_future["sample_elapsed"] - elapsed) if best_future else None
    elif phase == "avoidance":
        status = "Avoidance active"
        note = "Collision path is being corrected by thruster burn."
        time_to_collision = None
    elif phase == "recovery":
        status = "Recovery"
        note = "Closest approach has passed; satellite is returning to nominal orbit."
        time_to_collision = None
    else:
        status = "Stable"
        note = "Collision risk cleared."
        time_to_collision = None

    return {
        "status": status,
        "time_to_collision_s": round(time_to_collision, 1) if time_to_collision is not None else None,
        "distance_left_km": round(current["distance_units"] * km_per_unit, 1),
        "closest_approach_km": round(best_future["distance_units"] * km_per_unit, 1) if best_future else round(current["distance_units"] * km_per_unit, 1),
        "tracked_object": current["threat_id"],
        "note": note
    }

def current_phase():
    if not MISSION["started"] or MISSION["start_time"] is None:
        return "overview", 0.0
    t = time.time() - MISSION["start_time"]
    if t < 7:
        return "launch", t
    elif t < 14:
        return "deploy", t
    elif t < 24:
        return "orbit_nominal", t
    elif t < 34:
        return "threat", t
    elif t < 44:
        return "avoidance", t
    elif t < 54:
        return "recovery", t
    else:
        return "stable", t

@app.get("/")
def root():
    return RedirectResponse(url="/ui")

@app.get("/api/configure")
def api_configure(sats: int = 5, debris: int = 4, radius: float = 7700, topic: str = ""):
    CONFIG["sat_count"] = max(2, min(5, int(sats)))
    CONFIG["debris_count"] = max(1, int(debris))
    CONFIG["orbit_radius_km"] = float(radius)
    CONFIG["alert_topic"] = topic.strip()
    MISSION["started"] = False
    MISSION["start_time"] = None
    MISSION["alert_sent"] = False
    EVENTS.clear()
    log_event("info", f"Scenario configured with {CONFIG['sat_count']} satellites, {CONFIG['debris_count']} debris, orbit radius {CONFIG['orbit_radius_km']} km.")
    if CONFIG["alert_topic"]:
        log_event("info", f"Phone alert topic set to {CONFIG['alert_topic']}.")
    return {"status": "configured", **CONFIG}

@app.get("/api/start")
def api_start():
    MISSION["started"] = True
    MISSION["start_time"] = time.time()
    MISSION["alert_sent"] = False
    log_event("info", "Mission demo started.")
    return {"status": "started"}

@app.get("/api/reset")
def api_reset():
    MISSION["started"] = False
    MISSION["start_time"] = None
    MISSION["alert_sent"] = False
    EVENTS.clear()
    log_event("info", "Mission reset.")
    return {"status": "reset"}

@app.get("/api/test_phone_alert")
def api_test_phone_alert():
    result = send_alert("Orbital Insight Test", "Test notification from Orbital Insight.")
    if result.get("sent"):
        log_event("ok", f"Test phone alert sent to topic {result.get('topic')}.")
    else:
        log_event("warn", f"Phone alert failed: {result.get('reason')}")
    return result

@app.get("/api/state")
def api_state():
    phase, elapsed = current_phase()

    if phase == "launch" and not any(e["message"] == "Rocket launch initiated." for e in EVENTS):
        log_event("info", "Rocket launch initiated.")
    if phase == "deploy" and not any(e["message"] == "Satellites deployed into Earth orbit." for e in EVENTS):
        log_event("info", "Satellites deployed into Earth orbit.")
        log_event("warn", "Leftover rocket fragments remain in orbit as debris.")
    if phase == "threat" and not any(e["message"] == "Disconnected satellite and debris drifting toward protected satellite." for e in EVENTS):
        log_event("warn", "Disconnected satellite and debris drifting toward protected satellite.")
        if not MISSION["alert_sent"]:
            result = send_alert("Collision Alert", "Incoming orbital threat detected. Avoidance maneuver will begin.")
            MISSION["alert_sent"] = True
            if result.get("sent"):
                log_event("warn", "Phone alert sent to mission control.")
            else:
                log_event("warn", f"Phone alert failed: {result.get('reason')}")
    if phase == "avoidance" and not any(e["message"] == "Protected satellite executing thruster burn." for e in EVENTS):
        log_event("warn", "Protected satellite executing thruster burn.")
    if phase == "recovery" and not any(e["message"] == "Protected satellite returning to nominal orbit." for e in EVENTS):
        log_event("ok", "Protected satellite returning to nominal orbit.")
    if phase == "stable" and not any(e["message"] == "Mission stable after avoidance and orbit recovery." for e in EVENTS):
        log_event("ok", "Mission stable after avoidance and orbit recovery.")

    return JSONResponse({
        "config": CONFIG,
        "phase": phase,
        "elapsed": elapsed,
        "topic": CONFIG["alert_topic"],
        "collision": collision_metrics(phase, elapsed, CONFIG),
        "events": EVENTS[-30:]
    })

@app.get("/ui", response_class=HTMLResponse)
def ui():
    return """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Orbital Insight</title>
<style>
:root{
  --bg:#06101d; --panel:#0d1728; --panel2:#101d33; --line:rgba(255,255,255,.08);
  --text:#e8eefc; --muted:#9fb0d1; --blue:#2563eb; --blue2:#1d4ed8; --green:#22c55e;
  --yellow:#f59e0b; --red:#ef4444;
}
*{box-sizing:border-box}
html,body{margin:0;height:100%;background:var(--bg);color:var(--text);font-family:Arial,sans-serif}
.app{display:flex;height:100vh;overflow:hidden}
.sidebar{width:260px;background:#08101c;border-right:1px solid var(--line);display:flex;flex-direction:column}
.head{padding:16px;border-bottom:1px solid var(--line)}
.head h1{margin:0;font-size:22px}.head p{margin:4px 0 0;color:var(--muted);font-size:12px}
.nav{padding:14px;display:flex;flex-direction:column;gap:10px}
.nav button{width:100%;text-align:left;padding:14px 16px;border-radius:14px;border:1px solid var(--line);background:transparent;color:var(--text);cursor:pointer;font-size:15px;font-weight:600}
.nav button.active{background:linear-gradient(180deg,var(--blue),var(--blue2));border-color:transparent}
.main{flex:1;display:flex;flex-direction:column;min-width:0}
.top{height:72px;border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;padding:0 16px;background:#09101ccc}
.title{font-size:20px;font-weight:700}.sub{font-size:12px;color:var(--muted)}
.content{flex:1;overflow:auto;padding:16px}
.page{display:none}.page.active{display:block}
.grid{display:grid;grid-template-columns:2fr 1fr;gap:16px}
.two{display:grid;grid-template-columns:1fr 1fr;gap:16px}
.card{background:var(--panel);border:1px solid var(--line);border-radius:20px;overflow:hidden}
.pad{padding:16px}
.viewer{position:relative;height:78vh;background:#020814}
#spaceCanvas{width:100%;height:100%;display:block}
.overlay{position:absolute;top:14px;left:14px;right:14px;display:flex;justify-content:space-between;gap:10px;pointer-events:none}
.group{display:flex;gap:10px;flex-wrap:wrap}
.badge{padding:10px 14px;border-radius:999px;background:#0d1322d9;border:1px solid var(--line);font-size:13px;pointer-events:auto}
.toolbar{position:absolute;bottom:14px;left:14px;right:14px;display:flex;justify-content:space-between;gap:10px;pointer-events:none}
.toolbar .g{display:flex;gap:10px;flex-wrap:wrap;pointer-events:auto}
.btn{border:none;cursor:pointer;padding:11px 15px;border-radius:12px;font-size:14px;font-weight:600;color:#fff;background:linear-gradient(180deg,var(--blue),var(--blue2))}
.btn.secondary{background:linear-gradient(180deg,#475569,#334155)}
.btn.warn{background:linear-gradient(180deg,#f59e0b,#d97706)}
.metrics{display:grid;grid-template-columns:1fr 1fr;gap:12px}
.metric.wide{grid-column:1 / -1}
.metric{background:var(--panel2);border:1px solid var(--line);border-radius:16px;padding:14px}
.metric .label{color:var(--muted);font-size:12px}.metric .value{margin-top:6px;font-size:24px;font-weight:700}
.section-title{font-size:20px;font-weight:700;margin:18px 0 10px}.small{color:var(--muted);font-size:12px}
.log{display:flex;flex-direction:column;gap:10px;max-height:52vh;overflow:auto;padding-right:4px}
.log-item{background:var(--panel2);border:1px solid var(--line);border-left:4px solid var(--red);border-radius:14px;padding:12px;line-height:1.4}
.log-item.ok{border-left-color:var(--green)}.log-item.warn{border-left-color:var(--yellow)}
.form-grid{display:grid;grid-template-columns:1fr 1fr;gap:12px}
.field{display:flex;flex-direction:column;gap:8px}
.field label{color:var(--muted);font-size:12px}
.field input{width:100%;background:#0b1220;color:var(--text);border:1px solid var(--line);border-radius:12px;padding:12px;font-size:14px}
.row{display:flex;gap:10px;flex-wrap:wrap;margin-top:12px}
@media (max-width:1200px){.grid,.two{grid-template-columns:1fr}}
</style>
</head>
<body>
<div class="app">
  <aside class="sidebar">
    <div class="head">
      <h1>Orbital Insight</h1>
      <p>Final corrected build</p>
    </div>
    <div class="nav">
      <button id="nav-out" class="active" onclick="showPage('out')">Simulation Output</button>
      <button id="nav-in" onclick="showPage('in')">Scenario Builder</button>
      <button id="nav-safe" onclick="showPage('safe')">Safety Feed</button>
    </div>
  </aside>

  <main class="main">
    <div class="top">
      <div>
        <div class="title" id="pageTitle">Simulation Output</div>
        <div class="sub" id="pageSub">Smoother motion, rotating + revolving solar overview, Earth mission demo after start.</div>
      </div>
      <div class="small">Stable for Colab/mobile</div>
    </div>

    <div class="content">
      <div id="page-out" class="page active">
        <div class="grid">
          <div class="card viewer">
            <canvas id="spaceCanvas"></canvas>
            <div class="overlay">
              <div class="group">
                <div class="badge" id="phaseBadge">Phase: Overview</div>
                <div class="badge" id="modeBadge">Mode: Solar Overview</div>
              </div>
              <div class="group">
                <div class="badge">Drag to rotate • Scroll to zoom</div>
              </div>
            </div>
            <div class="toolbar">
              <div class="g">
                <button class="btn" onclick="configureScenario()">Load Scenario</button>
                <button class="btn" onclick="startMission()">Start Simulation</button>
                <button class="btn secondary" onclick="resetMission()">Reset</button>
                <button class="btn secondary" onclick="resetView()">Reset View</button>
              </div>
              <div class="g">
                <button class="btn warn" onclick="showPage('in')">Inputs</button>
              </div>
            </div>
          </div>

          <div class="card pad">
            <div class="metrics">
              <div class="metric"><div class="label">Satellites</div><div class="value" id="satMetric">5</div></div>
              <div class="metric"><div class="label">Debris</div><div class="value" id="debrisMetric">4</div></div>
              <div class="metric"><div class="label">Orbit Radius</div><div class="value" id="radiusMetric">7700</div></div>
              <div class="metric"><div class="label">Alert Topic</div><div class="value" id="topicMetric">None</div></div>
              <div class="metric"><div class="label">Time to Closest Approach</div><div class="value" id="timeLeftMetric">--</div></div>
              <div class="metric"><div class="label">Current Separation</div><div class="value" id="distanceMetric">--</div></div>
              <div class="metric wide"><div class="label">Collision Telemetry</div><div class="value" id="collisionStatusMetric" style="font-size:18px">Monitoring</div><div class="small" id="collisionNote">No imminent collision window yet.</div></div>
            </div>
            <div class="section-title">Latest Events</div>
            <div class="log" id="shortLog"></div>
          </div>
        </div>
      </div>

      <div id="page-in" class="page">
        <div class="two">
          <div class="card pad">
            <div class="section-title">Scenario Builder</div>
            <div class="small">Satellite count capped at 5.</div>
            <div class="form-grid">
              <div class="field"><label>Satellite count</label><input id="satCount" type="number" min="2" max="5" value="5"></div>
              <div class="field"><label>Debris count</label><input id="debrisCount" type="number" min="1" value="4"></div>
              <div class="field"><label>Orbit radius (km)</label><input id="orbitRadius" type="number" min="7000" value="7700"></div>
              <div class="field"><label>ntfy topic</label><input id="topicInput" type="text" placeholder="same topic as ntfy app"></div>
            </div>
            <div class="row">
              <button class="btn" onclick="configureScenario()">Configure</button>
              <button class="btn secondary" onclick="testPhoneAlert()">Test Phone Alert</button>
              <button class="btn warn" onclick="showPage('out')">Open Output</button>
            </div>
          </div>

          <div class="card pad">
            <div class="section-title">Demo flow</div>
            <div class="small">
              Before start: rotating + revolving solar overview.<br><br>
              After start: Earth mission mode.<br>
              1. Ground scene appears.<br>
              2. Rocket launches.<br>
              3. Satellites deploy into orbit.<br>
              4. Rocket leftovers remain as debris.<br>
              5. Threat approaches a protected satellite.<br>
              6. Thruster burn shifts the orbit.<br>
              7. Satellite returns to nominal orbit.
            </div>
          </div>
        </div>
      </div>

      <div id="page-safe" class="page">
        <div class="card pad">
          <div class="section-title">Safety Feed</div>
          <div class="log" id="fullLog"></div>
        </div>
      </div>
    </div>
  </main>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
<script>
let scene, camera, renderer, controls, canvas;
let sun, earth, moon, stars, groundGroup, rocket;
let solarBodies = [], solarOrbits = [];
let missionMeshes = {}, missionTargets = {}, missionTrails = {};
let riskLine = null, thrusterMesh = null;
let uiState = null;

function showPage(page){
  document.querySelectorAll('.page').forEach(p=>p.classList.remove('active'));
  document.querySelectorAll('.nav button').forEach(b=>b.classList.remove('active'));
  document.getElementById('page-'+page).classList.add('active');
  document.getElementById('nav-'+page).classList.add('active');
  const titles = {
    out: ['Simulation Output','Smoother motion, rotating + revolving solar overview, Earth mission demo after start.'],
    in: ['Scenario Builder','Configure the demo and phone alerts.'],
    safe: ['Safety Feed','Mission events and alerts.']
  };
  document.getElementById('pageTitle').innerText = titles[page][0];
  document.getElementById('pageSub').innerText = titles[page][1];
}

function buildScene(){
  canvas = document.getElementById('spaceCanvas');
  scene = new THREE.Scene();
  scene.background = new THREE.Color(0x020814);

  camera = new THREE.PerspectiveCamera(60, canvas.clientWidth / canvas.clientHeight, 0.1, 2000);
  camera.position.set(0,16,52);

  renderer = new THREE.WebGLRenderer({canvas: canvas, antialias: true});
  renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
  renderer.setSize(canvas.clientWidth, canvas.clientHeight, false);

  controls = new THREE.OrbitControls(camera, canvas);
  controls.enableDamping = true;
  controls.dampingFactor = 0.06;

  scene.add(new THREE.AmbientLight(0xffffff, 0.6));
  const point = new THREE.PointLight(0xffffff, 2.0, 0, 2);
  point.position.set(0,0,0);
  scene.add(point);

  const sg = new THREE.BufferGeometry();
  const count = 2600;
  const arr = new Float32Array(count*3);
  for (let i=0;i<count;i++){
    const r = 120 + Math.random()*220;
    const a = Math.random()*Math.PI*2;
    const b = Math.acos(Math.random()*2 - 1);
    arr[i*3] = r*Math.sin(b)*Math.cos(a);
    arr[i*3+1] = r*Math.sin(b)*Math.sin(a);
    arr[i*3+2] = r*Math.cos(b);
  }
  sg.setAttribute('position', new THREE.BufferAttribute(arr,3));
  stars = new THREE.Points(sg, new THREE.PointsMaterial({color:0xffffff, size:0.35}));
  scene.add(stars);

  sun = new THREE.Mesh(new THREE.SphereGeometry(3.3,40,40), new THREE.MeshPhongMaterial({color:0xffcc55}));
  scene.add(sun);

  buildSolarOverview();
  buildMissionAssets();
  window.addEventListener('resize', onResize);
  animate();
}

function buildSolarOverview(){
  const defs = [
    ['Mercury', 9, 0.50, 0x9e9e9e],
    ['Venus', 12, 0.75, 0xc68642],
    ['Earth', 16, 0.95, 0x2b6cb0],
    ['Mars', 20, 0.72, 0xb85c38],
    ['Jupiter', 26, 2.0, 0xcaa77a],
    ['Saturn', 33, 1.7, 0xd7c08a],
    ['Uranus', 39, 1.25, 0x8bd3dd],
    ['Neptune', 45, 1.2, 0x4f83ff],
  ];
  defs.forEach((d, idx)=>{
    const orbit = new THREE.LineLoop(
      new THREE.BufferGeometry().setFromPoints(
        Array.from({length:120}, (_,k)=>{
          const a = k/120 * Math.PI*2;
          return new THREE.Vector3(Math.cos(a)*d[1], 0, Math.sin(a)*d[1]);
        })
      ),
      new THREE.LineBasicMaterial({color:0x3f4c62, transparent:true, opacity:0.7})
    );
    scene.add(orbit);
    solarOrbits.push(orbit);

    const mesh = new THREE.Mesh(new THREE.SphereGeometry(d[2],28,28), new THREE.MeshPhongMaterial({color:d[3]}));
    mesh.userData = {radius:d[1], speed:0.12/(idx+1), phase:idx*0.8, isPlanet:true};
    scene.add(mesh);
    solarBodies.push(mesh);

    if (d[0] === 'Saturn'){
      const ring = new THREE.Mesh(
        new THREE.RingGeometry(d[2]*1.35, d[2]*2.0, 48),
        new THREE.MeshBasicMaterial({color:0xcab892, side:THREE.DoubleSide, transparent:true, opacity:0.8})
      );
      ring.userData = {follow: mesh};
      ring.rotation.x = Math.PI/2.8;
      scene.add(ring);
      solarBodies.push(ring);
    }
  });
}

function buildMissionAssets(){
  earth = new THREE.Mesh(new THREE.SphereGeometry(5,48,48), new THREE.MeshPhongMaterial({color:0x2b6cb0}));
  earth.visible = false; scene.add(earth);

  moon = new THREE.Mesh(new THREE.SphereGeometry(0.8,24,24), new THREE.MeshPhongMaterial({color:0xd1d5db}));
  moon.visible = false; scene.add(moon);

  groundGroup = new THREE.Group();
  groundGroup.visible = false;
  const ground = new THREE.Mesh(new THREE.CylinderGeometry(8,8,0.5,48), new THREE.MeshPhongMaterial({color:0x1f7a35}));
  ground.position.set(0,-8.6,0);
  groundGroup.add(ground);

  const tower = new THREE.Mesh(new THREE.BoxGeometry(0.6,5,0.6), new THREE.MeshPhongMaterial({color:0x6b7280}));
  tower.position.set(-1.2,-6.0,0);
  groundGroup.add(tower);

  rocket = new THREE.Group();
  const body = new THREE.Mesh(new THREE.CylinderGeometry(0.22,0.26,1.8,16), new THREE.MeshPhongMaterial({color:0xe5e7eb}));
  body.rotation.z = Math.PI/2; rocket.add(body);
  const nose = new THREE.Mesh(new THREE.ConeGeometry(0.22,0.48,16), new THREE.MeshPhongMaterial({color:0xef4444}));
  nose.rotation.z = -Math.PI/2; nose.position.set(1.15,0,0); rocket.add(nose);
  rocket.position.set(0,-7.4,0);
  groundGroup.add(rocket);
  scene.add(groundGroup);
}

function satelliteMesh(connected){
  const g = new THREE.Group();
  const body = new THREE.Mesh(new THREE.BoxGeometry(0.8,0.4,0.4), new THREE.MeshPhongMaterial({color:connected?0xffffff:0xff6666}));
  g.add(body);
  const panelMat = new THREE.MeshPhongMaterial({color:0x60a5fa});
  const left = new THREE.Mesh(new THREE.BoxGeometry(0.9,0.07,0.32), panelMat);
  left.position.set(-0.95,0,0); g.add(left);
  const right = left.clone(); right.position.x = 0.95; g.add(right);
  return g;
}

function debrisMesh(size){
  return new THREE.Mesh(new THREE.DodecahedronGeometry(size || 0.26,0), new THREE.MeshPhongMaterial({color:0x8b5a2b, flatShading:true}));
}

function ensureMissionMesh(id, type, connected, size){
  if (missionMeshes[id]) return missionMeshes[id];
  const mesh = type === 'sat' ? satelliteMesh(connected) : debrisMesh(size);
  scene.add(mesh);
  const trail = new THREE.Line(
    new THREE.BufferGeometry().setFromPoints([new THREE.Vector3(), new THREE.Vector3()]),
    new THREE.LineBasicMaterial({color:0xcbd5e1, transparent:true, opacity:0.6})
  );
  scene.add(trail);
  missionMeshes[id] = mesh;
  missionTargets[id] = new THREE.Vector3();
  missionTrails[id] = {line: trail, pts: []};
  return mesh;
}

function clearMissionVisuals(){
  Object.keys(missionMeshes).forEach(id => {
    scene.remove(missionMeshes[id]);
    scene.remove(missionTrails[id].line);
  });
  missionMeshes = {};
  missionTargets = {};
  missionTrails = {};
  if (riskLine) { scene.remove(riskLine); riskLine = null; }
  if (thrusterMesh) { scene.remove(thrusterMesh); thrusterMesh = null; }
}

function updateTrail(id){
  const mesh = missionMeshes[id], t = missionTrails[id];
  const pos = mesh.position.clone();
  t.pts.push(pos);
  if (t.pts.length > 120) t.pts.shift();
  t.line.geometry.setFromPoints(t.pts);
}

function updateOverview(timeSec){
  sun.visible = true;
  solarOrbits.forEach(o => o.visible = true);
  solarBodies.forEach(p => p.visible = true);
  earth.visible = false; moon.visible = false; groundGroup.visible = false;
  clearMissionVisuals();

  sun.rotation.y += 0.003;

  solarBodies.forEach((p) => {
    if (p.userData && p.userData.isPlanet){
      const angle = timeSec * p.userData.speed + p.userData.phase;
      p.position.set(Math.cos(angle)*p.userData.radius, 0, Math.sin(angle)*p.userData.radius);
      p.rotation.y += 0.01;
    } else if (p.userData && p.userData.follow) {
      p.position.copy(p.userData.follow.position);
      p.rotation.z += 0.002;
    }
  });
}

function missionObjectsForPhase(phase, elapsed, config){
  const out = [];
  const satCount = Math.min(5, Math.max(2, parseInt(config.sat_count || 5)));
  const connected = Math.max(1, satCount - 2);
  const disconnected = satCount - connected;

  for (let i=0;i<connected;i++){
    const a = (i/connected) * Math.PI*2 + elapsed*0.23;
    out.push({id:'SAT-C'+(i+1), type:'sat', connected:true, x:Math.cos(a)*(9 + (i%2)*0.5), y:0, z:Math.sin(a)*(9 + (i%2)*0.5), burn:false, recover:false});
  }
  for (let i=0;i<disconnected;i++){
    const a = 1.2 + i*1.4 + elapsed*0.19;
    let r = 10.2;
    if (phase==='threat' || phase==='avoidance' || phase==='recovery' || phase==='stable') r -= Math.min(2.5, (elapsed-24)*0.11);
    out.push({id:'SAT-D'+(i+1), type:'sat', connected:false, x:Math.cos(a)*r, y:0, z:Math.sin(a)*r, burn:false, recover:false});
  }
  for (let i=0;i<Math.max(1, parseInt(config.debris_count || 4));i++){
    const a = 0.8 + i*0.9 + elapsed*(0.16+i*0.01);
    let r = 11.2 + i*0.35;
    if (phase==='threat' || phase==='avoidance' || phase==='recovery' || phase==='stable') r -= Math.min(3.0, (elapsed-24)*0.08);
    out.push({id:'DEB-'+(i+1), type:'debris', connected:false, x:Math.cos(a)*r, y:0, z:Math.sin(a)*r, size:0.22 + i*0.03, burn:false, recover:false});
  }

  const protectedSat = out.find(o => o.id==='SAT-C1');
  if (protectedSat){
    if (phase==='avoidance'){
      protectedSat.y = Math.min(2.2, (elapsed-34)*0.42);
      protectedSat.burn = true;
    }
    if (phase==='recovery'){
      protectedSat.y = Math.max(0, 2.2 - (elapsed-44)*0.24);
      protectedSat.recover = true;
    }
  }
  return out;
}

function updateMission(phase, elapsed, config){
  sun.visible = false;
  solarOrbits.forEach(o => o.visible = false);
  solarBodies.forEach(p => p.visible = false);

  earth.visible = true;
  moon.visible = true;
  groundGroup.visible = (phase === 'launch' || phase === 'deploy');
  earth.rotation.y += 0.001;
  moon.position.set(Math.cos(elapsed*0.55)*12, 0, Math.sin(elapsed*0.55)*12);

  const objs = missionObjectsForPhase(phase, elapsed, config);

  if (phase === 'launch'){
    rocket.visible = true;
    rocket.position.set(0, -7.4 + elapsed*0.75, 0);
  } else if (phase === 'deploy'){
    rocket.visible = true;
    rocket.position.set(1.5 + (elapsed-7)*0.25, -2.0 + (elapsed-7)*0.2, 0);
  } else {
    rocket.visible = false;
  }

  const validIds = new Set();
  objs.forEach(o => {
    validIds.add(o.id);
    const mesh = ensureMissionMesh(o.id, o.type, !!o.connected, o.size || 0.26);
    missionTargets[o.id].set(o.x, o.y, o.z);

    if (o.type === 'sat'){
      const body = mesh.children[0];
      if (o.burn) body.material.color.setHex(0xffff66);
      else if (o.recover) body.material.color.setHex(0x22c55e);
      else if (!o.connected) body.material.color.setHex(0xff6666);
      else body.material.color.setHex(0xffffff);
    }
  });

  Object.keys(missionMeshes).forEach(id => {
    if (!validIds.has(id)){
      scene.remove(missionMeshes[id]);
      scene.remove(missionTrails[id].line);
      delete missionMeshes[id];
      delete missionTargets[id];
      delete missionTrails[id];
    }
  });

  Object.keys(missionMeshes).forEach(id => {
    missionMeshes[id].position.lerp(missionTargets[id], 0.12);
    updateTrail(id);
  });

  if (riskLine) { scene.remove(riskLine); riskLine = null; }
  if (thrusterMesh) { scene.remove(thrusterMesh); thrusterMesh = null; }

  if (phase === 'threat' || phase === 'avoidance' || phase === 'recovery'){
    const protectedSat = objs.find(o => o.id === 'SAT-C1');
    const threat = objs.find(o => o.id === 'SAT-D1') || objs.find(o => o.id === 'DEB-1');
    if (protectedSat && threat){
      riskLine = new THREE.Line(
        new THREE.BufferGeometry().setFromPoints([
          missionTargets[protectedSat.id].clone(),
          new THREE.Vector3(threat.x, threat.y, threat.z)
        ]),
        new THREE.LineBasicMaterial({color:0xff4444, transparent:true, opacity:0.9})
      );
      scene.add(riskLine);
    }
    if (phase === 'avoidance' && protectedSat){
      thrusterMesh = new THREE.ArrowHelper(
        new THREE.Vector3(0,1,0),
        missionTargets[protectedSat.id].clone().add(new THREE.Vector3(0,-0.5,0)),
        2.0,
        0x67e8f9,
        0.45,
        0.25
      );
      scene.add(thrusterMesh);
    }
  }

  if (phase === 'launch'){
    camera.position.lerp(new THREE.Vector3(0,-1,16), 0.04);
  } else if (phase === 'deploy'){
    camera.position.lerp(new THREE.Vector3(7,6,18), 0.04);
  } else if (phase === 'orbit_nominal'){
    camera.position.lerp(new THREE.Vector3(0,12,24), 0.04);
  } else if (phase === 'threat' || phase === 'avoidance' || phase === 'recovery'){
    camera.position.lerp(new THREE.Vector3(10,8,14), 0.04);
  } else if (phase === 'stable'){
    camera.position.lerp(new THREE.Vector3(0,12,24), 0.03);
  }
  controls.target.lerp(new THREE.Vector3(0,0,0), 0.06);
}

async function refreshUI(){
  const r = await fetch('/api/state');
  uiState = await r.json();

  document.getElementById('satMetric').innerText = uiState.config.sat_count;
  document.getElementById('debrisMetric').innerText = uiState.config.debris_count;
  document.getElementById('radiusMetric').innerText = Math.round(uiState.config.orbit_radius_km);
  document.getElementById('topicMetric').innerText = uiState.topic || 'None';
  document.getElementById('phaseBadge').innerText = 'Phase: ' + (uiState.phase === 'overview' ? 'Overview' : uiState.phase);
  document.getElementById('modeBadge').innerText = 'Mode: ' + (uiState.phase === 'overview' ? 'Solar Overview' : 'Earth Mission Demo');

  const collision = uiState.collision || {};
  document.getElementById('timeLeftMetric').innerText = collision.time_to_collision_s == null ? 'Avoided' : (collision.time_to_collision_s.toFixed(1) + ' s');
  document.getElementById('distanceMetric').innerText = collision.distance_left_km == null ? '--' : (collision.distance_left_km.toFixed(1) + ' km');
  document.getElementById('collisionStatusMetric').innerText = collision.status || 'Monitoring';
  document.getElementById('collisionNote').innerText = (collision.tracked_object ? ('Tracking ' + collision.tracked_object + '. ') : '') + (collision.note || '');

  const html = (uiState.events || []).slice().reverse().map(e => {
    const cls = e.kind === 'warn' ? 'warn' : e.kind === 'ok' ? 'ok' : '';
    return `<div class="log-item ${cls}"><div><b>${e.kind.toUpperCase()}</b></div><div>${e.message}</div></div>`;
  }).join('');
  document.getElementById('shortLog').innerHTML = html || '<div class="log-item ok">No events yet.</div>';
  document.getElementById('fullLog').innerHTML = html || '<div class="log-item ok">No events yet.</div>';
}

async function configureScenario(){
  const sats = Math.min(5, Math.max(2, parseInt(document.getElementById('satCount').value || '5')));
  const debris = Math.max(1, parseInt(document.getElementById('debrisCount').value || '4'));
  const radius = parseFloat(document.getElementById('orbitRadius').value || '7700');
  const topic = document.getElementById('topicInput').value.trim();
  await fetch(`/api/configure?sats=${sats}&debris=${debris}&radius=${radius}&topic=${encodeURIComponent(topic)}`);
  await refreshUI();
  showPage('out');
}

async function startMission(){
  await fetch('/api/start');
  await refreshUI();
}

async function resetMission(){
  await fetch('/api/reset');
  await refreshUI();
  camera.position.set(0,16,52);
  controls.target.set(0,0,0);
  controls.update();
}

async function testPhoneAlert(){
  const topic = document.getElementById('topicInput').value.trim();
  if (topic){
    const sats = Math.min(5, Math.max(2, parseInt(document.getElementById('satCount').value || '5')));
    const debris = Math.max(1, parseInt(document.getElementById('debrisCount').value || '4'));
    const radius = parseFloat(document.getElementById('orbitRadius').value || '7700');
    await fetch(`/api/configure?sats=${sats}&debris=${debris}&radius=${radius}&topic=${encodeURIComponent(topic)}`);
  }
  const r = await fetch('/api/test_phone_alert');
  const data = await r.json();
  alert(data.sent ? ('Alert sent to topic: ' + data.topic) : ('Alert failed: ' + (data.reason || 'unknown')));
  await refreshUI();
}

function resetView(){
  if (!uiState || uiState.phase === 'overview'){
    camera.position.set(0,16,52);
  } else {
    camera.position.set(0,12,24);
  }
  controls.target.set(0,0,0);
  controls.update();
}

function onResize(){
  const w = canvas.clientWidth, h = canvas.clientHeight;
  camera.aspect = w/h;
  camera.updateProjectionMatrix();
  renderer.setSize(w,h,false);
}

function animate(){
  requestAnimationFrame(animate);
  if (controls) controls.update();
  const t = performance.now() * 0.001;
  if (!uiState || uiState.phase === 'overview'){
    updateOverview(t);
  } else {
    updateMission(uiState.phase, uiState.elapsed, uiState.config);
  }
  if (stars) stars.rotation.y += 0.00005;
  renderer.render(scene, camera);
}

buildScene();
refreshUI();
setInterval(refreshUI, 1500);
</script>
</body>
</html>
"""

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

'''
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)
print('app.py written')

app.py written


In [3]:
import nest_asyncio
nest_asyncio.apply()

In [4]:

import subprocess, time
server_process = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)
print("Server started on http://127.0.0.1:8000")


Server started on http://127.0.0.1:8000


In [5]:

import requests
for url in ["http://127.0.0.1:8000/", "http://127.0.0.1:8000/api/state", "http://127.0.0.1:8000/ui"]:
    r = requests.get(url, timeout=10)
    print(url, "->", r.status_code)


http://127.0.0.1:8000/ -> 200
http://127.0.0.1:8000/api/state -> 200
http://127.0.0.1:8000/ui -> 200


In [6]:

import os, re, stat, subprocess, time, urllib.request

CF_BIN = "/content/cloudflared"
if not os.path.exists(CF_BIN):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF_BIN,
    )
    os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IEXEC)

cf_process = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
start = time.time()
while time.time() - start < 30:
    line = cf_process.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    print(line, end="")
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\nPublic URL:", public_url)
    print("Open:", public_url + "/ui")
else:
    print("Cloudflare Tunnel started, but the public URL was not detected automatically.")
    print("Check the cell output above for the trycloudflare.com URL.")


2026-03-27T21:08:22Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-27T21:08:22Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-27T21:08:25Z INF +--------------------------------------------------------------------------------------------+
2026-03-27T21:08:25Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-27T21:08:25Z INF |  https://photographer-zones-manufacture-folders.tryclo